In [12]:
from manim import *
import numpy as np
# Removed the failing import: from manim.utils.rate_functions import slow_into_fast

class TSNEExplanation(Scene):
    def construct(self):
        # --- FIX: Attempt to set scale factor directly on the camera object (Handling previous errors) ---
        try:
            self.camera.scale_factor = 1/0.9
        except AttributeError:
            # Fallback: Ignore scaling if the method isn't supported by this Manim version
            pass 
        
        # --- 0. Title and Overview ---
        title = Text("t-SNE: t-distributed Stochastic Neighbor Embedding", font_size=40).to_edge(UP)
        self.add(title) 
        self.wait(0.5)
        
        # Initial Subtitle for smooth start
        self.current_subtitle = Text("Introduction to Dimensionality Reduction", font_size=32).next_to(title, DOWN, buff=0.4).to_edge(LEFT)
        self.play(Write(self.current_subtitle))
        self.wait(1)
        
        # --- 1. High-Dimensional Similarity (P_ij) ---
        self.section_1_high_d_similarity()
        
        # --- 2. Low-Dimensional Similarity (Q_ij) ---
        self.section_2_low_d_similarity()
        
        # --- 3. The Objective Function (KL Divergence) ---
        self.section_3_kl_divergence()
        
        # --- 4. Minimization and Final Result ---
        self.section_4_minimization()
        
        # Final clear up
        self.play(FadeOut(title, self.current_subtitle))
        self.wait(1)

    # --- Helper function for smooth subtitle transitions ---
    def transition_subtitle(self, new_text):
        new_subtitle = Text(new_text, font_size=32).next_to(self.mobjects[0], DOWN, buff=0.4).to_edge(LEFT)
        self.play(
            ReplacementTransform(self.current_subtitle, new_subtitle)
        )
        self.current_subtitle = new_subtitle
        return new_subtitle

    # --- Section 1: High-Dimensional Similarity ---
    def section_1_high_d_similarity(self):
        
        self.transition_subtitle("1. High-D Similarity: Gaussian Distribution (P)")
        
        # Setup High-D points visualization
        X_group_A = VGroup(*[Dot(radius=0.08, color=BLUE).move_to([np.random.rand()*1.5 - 4.5, np.random.rand()*1.5 - 1, 0]) for _ in range(10)])
        X_group_B = VGroup(*[Dot(radius=0.08, color=RED).move_to([np.random.rand()*1.5 + 0.5, np.random.rand()*1.5 + 1, 0]) for _ in range(10)])
        X_points = VGroup(X_group_A, X_group_B).move_to(LEFT * 4 + DOWN * 0.5) # MOVED POINTS FURTHER LEFT
        high_d_label = Text("High-D Space (X)", font_size=20).next_to(X_points, UP)
        
        self.play(FadeIn(X_points), Write(high_d_label))
        
        # Focus on one point x_i
        x_i = X_group_A[0].copy().set_color(YELLOW).scale(1.5)
        self.play(Transform(X_group_A[0], x_i))
        x_i_label = MathTex("x_i").next_to(x_i, RIGHT, buff=0.1).set_color(YELLOW)
        self.play(FadeIn(x_i_label))

        # Show the Gaussian distribution and formula
        gaussian = Ellipse(width=3, height=2, color=YELLOW, fill_opacity=0.1).move_to(x_i.get_center())
        self.play(Create(gaussian))

        # [Image of Gaussian curve showing variance]

        
        p_ji_formula = MathTex(
            "p_{j|i} = ", 
            r"\frac{\exp\left(-\|x_i - x_j\|^2 / (2\sigma_i^2)\right)}{\sum_{k \neq i} \exp\left(-\|x_i - x_k\|^2 / (2\sigma_i^2)\right)}"
        ).scale(0.75).move_to(RIGHT * 3.75 + UP*1) # MOVED FORMULA FURTHER RIGHT
        self.play(Write(p_ji_formula))

        # Aligned right to use space efficiently
        perplexity_text = MathTex(r"\sigma_i \text{determined by Perplexity}", font_size=24, color=YELLOW).next_to(p_ji_formula, DOWN, aligned_edge=RIGHT, buff=0.5)
        self.play(Write(perplexity_text))
        
        self.wait(1)
        
        # Joint probability P_ij (Aligned right)
        p_ij_formula = MathTex(
            "p_{ij} = \\frac{p_{j|i} + p_{i|j}}{2N}", 
        ).scale(0.8).next_to(perplexity_text, DOWN, aligned_edge=RIGHT, buff=0.7) 
        
        self.play(Write(p_ij_formula))
        self.wait(1)
        
        # Group all temporary elements for clean exit
        section_1_group = VGroup(
            X_points, high_d_label, gaussian, x_i, x_i_label, 
            perplexity_text, p_ji_formula, p_ij_formula
        )
        
        self.play(
            FadeOut(section_1_group, run_time=1.5)
        )

    # --- Section 2: Low-Dimensional Similarity ---
    def section_2_low_d_similarity(self):
        
        self.transition_subtitle("2. Low-D Similarity: Student's t-Distribution (Q)")
        
        # Initial random embedding Y (Moved slightly left)
        Y_points = VGroup(*[Dot(radius=0.08, color=GRAY).move_to([np.random.rand()*5 - 2.5, np.random.rand()*3 - 2, 0]) for _ in range(40)])
        y_label = Text("Initial Low-D Space (Y)", font_size=20).next_to(Y_points, UP)
        self.play(FadeIn(Y_points), Write(y_label))
        
        # Show the Q_ij formula (heavy-tailed)
        q_ij_formula = MathTex(
            "q_{ij} = \\frac{(1 + ||y_i - y_j||^2)^{-1}}{\\sum_{k \\neq l} (1 + ||y_k - y_l||^2)^{-1}}"
        ).scale(0.7).to_edge(RIGHT).shift(UP*0.5)
        
        q_ij_box = SurroundingRectangle(q_ij_formula, color=PURPLE, buff=0.2)
        # Increased buff for separation
        heavy_tail_text = Text("Heavy Tail (t-dist): Solves Crowding", font_size=24, color=PURPLE).next_to(q_ij_box, DOWN, buff=0.4) 
        # 
        self.play(
            Write(q_ij_formula),
            Create(q_ij_box),
            Write(heavy_tail_text)
        )
        self.wait(2)
        
        # Group all temporary elements for clean exit
        section_2_group = VGroup(Y_points, y_label, q_ij_formula, q_ij_box, heavy_tail_text)
        
        self.play(
            FadeOut(section_2_group, run_time=1.5)
        )


    # --- Section 3: The Objective Function ---
    def section_3_kl_divergence(self):
        
        self.transition_subtitle("3. Objective: Minimize KL Divergence")
        
        kl_formula = MathTex(
            "C = \\text{KL}(P \\| Q) = \\sum_{i} \\sum_{j \\neq i} p_{ij} \log \\left(\\frac{p_{ij}}{q_{ij}}\\right)"
        ).scale(0.9).move_to(UP*1.2)
        
        self.play(Write(kl_formula))
        self.wait(0.5)
        # 

        # Illustrate forces
        # Increased buff for separation from kl_formula
        case_1_text = Text("High P / Low Q → Attraction", font_size=28, color=GREEN).next_to(kl_formula, DOWN, buff=1.5).to_edge(LEFT, buff=0.5)
        case_1_points = VGroup(
            Dot(color=GREEN).move_to(LEFT*2.5 + DOWN*2.5), 
            Dot(color=GREEN).move_to(LEFT*1.5 + DOWN*2.5)
        )
        case_1_arrow = Arrow(case_1_points[0].get_center(), case_1_points[1].get_center(), buff=0.3, color=GREEN).set_opacity(0.8)
        
        self.play(Write(case_1_text), FadeIn(case_1_points), FadeIn(case_1_arrow))
        
        # Simulate movement
        self.play(
            case_1_points[0].animate.shift(RIGHT*0.15),
            case_1_points[1].animate.shift(LEFT*0.15),
            case_1_arrow.animate.set_opacity(0.1),
            run_time=0.5
        )
        self.wait(0.5)

        # Case 2: Repulsion (p_ij low, q_ij high)
        case_2_text = Text("Low P / High Q → Repulsion", font_size=28, color=RED).next_to(kl_formula, DOWN, buff=1.5).to_edge(RIGHT, buff=0.5)
        case_2_points = VGroup(
            Dot(color=RED).move_to(RIGHT*1.5 + DOWN*2.5), 
            Dot(color=RED).move_to(RIGHT*2.5 + DOWN*2.5)
        )
        case_2_arrow = Arrow(case_2_points[0].get_center(), case_2_points[1].get_center(), buff=0.3, color=RED).set_opacity(0.8)

        self.play(Write(case_2_text), FadeIn(case_2_points), FadeIn(case_2_arrow))
        
        # Simulate movement
        self.play(
            case_2_points[0].animate.shift(LEFT*0.15),
            case_2_points[1].animate.shift(RIGHT*0.15),
            case_2_arrow.animate.set_opacity(0.1),
            run_time=0.5
        )
        self.wait(0.5)
        
        # Group all temporary elements for clean exit
        section_3_group = VGroup(
            kl_formula, case_1_text, case_1_points, case_1_arrow, 
            case_2_text, case_2_points, case_2_arrow
        )
        
        self.play(
            FadeOut(section_3_group, run_time=1.5)
        )


    # --- Section 4: Minimization and Final Result ---
    def section_4_minimization(self):
        
        self.transition_subtitle("4. Minimization: Gradient Descent")

        grad_formula = MathTex(
            r"\frac{\partial C}{\partial y_i} = 4 \sum_{j \neq i} (p_{ij} - q_{ij}) (1 + \|y_i - y_j\|^2)^{-1} (y_i - y_j)"
        ).scale(0.75).move_to(UP*1.2)
        
        self.play(Write(grad_formula))
        self.wait(1)

        # Increased buff for separation
        update_rule = MathTex(
            r"y_i^{(t)} = y_i^{(t-1)} + \eta \frac{\partial C}{\partial y_i} + \alpha (\text{Momentum})"
        ).scale(0.8).next_to(grad_formula, DOWN, buff=0.7)
        self.play(Write(update_rule))
        self.wait(1)
        
        self.play(FadeOut(grad_formula, update_rule))

        # --- Visualization of Optimization ---
        # Initial points (randomly scattered)
        initial_Y = VGroup(*[Dot(radius=0.08, color=GRAY).move_to([np.random.rand()*5 - 2.5, np.random.rand()*5 - 1, 0]) for _ in range(40)])
        
        # Final points (clustered result)
        final_Y_A = VGroup(*[Dot(radius=0.08, color=BLUE).move_to([np.random.rand()*1 + 1, np.random.rand()*1 - 1.5, 0]) for _ in range(20)])
        final_Y_B = VGroup(*[Dot(radius=0.08, color=RED).move_to([np.random.rand()*1 - 2, np.random.rand()*1 + 0.5, 0]) for _ in range(20)])
        final_Y = VGroup(final_Y_A, final_Y_B).move_to(DOWN*1.5)

        optimization_text = Text("Optimization: Points move to minimize cost C", font_size=28).move_to(UP*1)
        
        self.play(
            Write(optimization_text),
            FadeIn(initial_Y.shift(DOWN*0.5))
        )
        
        # Animate the transition from scattered to clustered
        self.play(
            Transform(initial_Y, final_Y),
            run_time=3,
            rate_func=smooth 
        )
        # 
        
        final_label = Text("Final t-SNE Embedding (Local Structure Preserved)", font_size=28, color=GREEN).next_to(final_Y, DOWN, buff=0.5)
        self.play(Write(final_label))
        
        # Group all temporary elements for clean exit
        section_4_group = VGroup(
            optimization_text, initial_Y, final_label
        )
        
        self.play(
            FadeOut(section_4_group, run_time=1.5)
        )




%manim -qk -v warning TSNEExplanation

Manim Community v0.19.0